In [ ]:
import boto3
from ./consts/consts_api import *
from .utils/utils_api import get_model_response

In [ ]:
bedrock_client = boto3.client(service_name="bedrock_runtime", region_name="us-west-2")
model_id = MODEL_ID

In [ ]:
def web_search(topic):
    print(f"pretending to search the web for {topic}")

web_search_tool = {
    "toolSpec": {
        "name": "web_search",
        "description": "A tool to retrieve up to date information on a given topic by searching the web.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search the web for, formatted as a web search query"
                    },
                },
                "required": ["search_term"]
            }
        }
    }
}

## Auto

In [ ]:
from datetime import date

def chat_with_web_search(user_query):

    system_prompt=f"""
    Answer as many questions as you can using your existing knowledge.
    Only seasrch the web for queries that you can not confidently answer.
    Today's date is {date.today().strtime(%B %d %y)}
    If you think a user's question involves something in the future that hasn't happend yet, use the search tool.
    """

    messages = [{"role": "user", "content": [{"text": user_query}]}]

    inference_config={"maxTokens":400}
    tool_config = {"tools":[web_search_tool],"toolChoice": {"auto":{}}}

    # Send the message.
    response = bedrock_client.converse(
        modelId=model_id,
        messages=messages,
        system=[{"text":system_prompt}],
        inferenceConfig=inference_config,
        toolConfig=tool_config,
    )

    last_content_block=response["output"]["message"]["content"][-1]
    if "text" in last_content_block:
        print("Claude did NOT call a tool!")
        print(f"Assistant: {last_content_block["text"]}")
    if "toolUse" in last_content_block:
        print("Claude wants to use a tool")
        print(last_content_block)

In [ ]:
chat_with_web_search("what does Palantir company do and what's his role in AI industry?")

## Forcing a Tool

In [ ]:
from datetime import date

def chat_with_web_search_forced_tool_use(user_query):

    system_prompt=f"""
    Search the web and respond to user's questions. You may not always need to call the tool.
    """

    messages = [{"role": "user", "content": [{"text": user_query}]}]

    inference_config={"maxTokens":400}
    tool_config = {"tools":[web_search_tool],"toolChoice": {"tool":{"name":"web_search"}}}

    # Send the message.
    response = bedrock_client.converse(
        modelId=model_id,
        messages=messages,
        system=[{"text":system_prompt}],
        inferenceConfig=inference_config,
        toolConfig=tool_config,
    )

    last_content_block=response["output"]["message"]["content"][-1]
    if "text" in last_content_block:
        print("Claude did NOT call a tool!")
        print(f"Assistant: {last_content_block["text"]}")
    if "toolUse" in last_content_block:
        print("Claude wants to use a tool")
        print(last_content_block)

In [ ]:
chat_with_web_search_forced_tool_use("What are the main AI companies that the majority of market cap in the technology industry")

## Any